# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

In [4]:
import nltk
nltk.download('gutenberg')
import re
from nltk.corpus import gutenberg
# 1. Cargar los textos originales
file_ids = gutenberg.fileids()
print(f"Número de documentos: {len(file_ids)}")

raw_docs = {fid: gutenberg.raw(fid) for fid in file_ids}

# 2. Tamaño de cada documento (caracteres y palabras aproximadas)
for fid in file_ids:
    n_chars = len(raw_docs[fid])
    n_words_approx = len(raw_docs[fid].split())
    print(f"{fid:30s} | caracteres: {n_chars:8d} | palabras aprox.: {n_words_approx:7d}")

# 3. Tokenización y normalización
def normalizar_y_tokenizar(texto):
    """
    Configuración de preprocesamiento utilizada de forma consistente
    en todo el notebook:
      - Conversión a minúsculas.
      - Eliminación de puntuación, dígitos y caracteres especiales
        (se conservan solo letras).
      - Tokenización por espacios una vez limpio el texto.
    """
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü]+", " ", texto)
    tokens = texto.split()
    return tokens

documentos_tokenizados = {fid: normalizar_y_tokenizar(raw_docs[fid]) for fid in file_ids}

# Verificación rápida
ejemplo = file_ids[0]
print(f"\nEjemplo de tokens de '{ejemplo}':")
print(documentos_tokenizados[ejemplo][:20])
print(f"Total de tokens en '{ejemplo}': {len(documentos_tokenizados[ejemplo])}")

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Jhonn22\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Número de documentos: 18
austen-emma.txt                | caracteres:   887071 | palabras aprox.:  158167
austen-persuasion.txt          | caracteres:   466292 | palabras aprox.:   83308
austen-sense.txt               | caracteres:   673022 | palabras aprox.:  118675
bible-kjv.txt                  | caracteres:  4332554 | palabras aprox.:  821133
blake-poems.txt                | caracteres:    38153 | palabras aprox.:    6845
bryant-stories.txt             | caracteres:   249439 | palabras aprox.:   45988
burgess-busterbrown.txt        | caracteres:    84663 | palabras aprox.:   15870
carroll-alice.txt              | caracteres:   144395 | palabras aprox.:   26443
chesterton-ball.txt            | caracteres:   457450 | palabras aprox.:   81598
chesterton-brown.txt           | caracteres:   406629 | palabras aprox.:   71626
chesterton-thursday.txt        | caracteres:   320525 | palabras aprox.:   57955
edgeworth-parents.txt          | caracteres:   935158 | palabras aprox.:  166070
mel

**Configuración de preprocesamiento elegida (documentación)**

- *Minúsculas*: se normaliza todo el texto a minúsculas para que `Whale` y `whale` cuenten como el mismo término.
- *Puntuación y caracteres especiales*: se eliminan con una expresión regular que conserva solo letras (`a-z` y vocales acentuadas/ñ), de modo que signos de puntuación, números y saltos de línea no generen tokens espurios.
- *Tokenización*: al quedar el texto limpio, se separa simplemente por espacios en blanco (`str.split()`), que es suficiente dado que ya no hay puntuación pegada a las palabras.
- Esta misma función `normalizar_y_tokenizar` se reutiliza sin cambios en la Parte B (corpus grande), tal como pide el enunciado.

**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [5]:
def busqueda_secuencial(termino, documentos_tokenizados):
    """
    Búsqueda secuencial de 'termino' en la colección.
    Dos lazos for anidados obligatorios:
      - El primero recorre los documentos.
      - El segundo recorre los tokens de cada documento.
    Devuelve la lista de ids de documentos donde aparece el término.
    """
    termino = termino.lower()
    resultados = []
    for doc_id, tokens in documentos_tokenizados.items():   # 1er lazo: documentos
        encontrado = False
        for token in tokens:                                 # 2do lazo: tokens
            if token == termino:
                encontrado = True
                break
        if encontrado:
            resultados.append(doc_id)
    return resultados

# Términos de prueba: frecuente, poco frecuente e inexistente
terminos_prueba = ["the", "whale", "xyzxyz"]

for t in terminos_prueba:
    r = busqueda_secuencial(t, documentos_tokenizados)
    print(f"'{t}': {len(r)} documento(s) -> {r}")

'the': 18 documento(s) -> ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
'whale': 6 documento(s) -> ['bible-kjv.txt', 'bryant-stories.txt', 'chesterton-ball.txt', 'melville-moby_dick.txt', 'shakespeare-hamlet.txt', 'whitman-leaves.txt']
'xyzxyz': 0 documento(s) -> []


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [7]:
import time

def medir_tiempo_secuencial(termino, documentos_tokenizados, n_ejecuciones=30):
    """Promedia el tiempo (en ms) de 30 ejecuciones de busqueda_secuencial."""
    tiempos_ns = []
    for _ in range(n_ejecuciones):
        inicio = time.perf_counter_ns()
        busqueda_secuencial(termino, documentos_tokenizados)
        fin = time.perf_counter_ns()
        tiempos_ns.append(fin - inicio)
    media_ns = sum(tiempos_ns) / len(tiempos_ns)
    return media_ns / 1e6  # ns -> ms

resultados_tiempo_secuencial = {}
for t in terminos_prueba:
    media_ms = medir_tiempo_secuencial(t, documentos_tokenizados)
    resultados_tiempo_secuencial[t] = media_ms
    print(f"'{t}': tiempo medio = {media_ms:.4f} ms (30 ejecuciones)")

'the': tiempo medio = 0.0146 ms (30 ejecuciones)
'whale': tiempo medio = 50.9040 ms (30 ejecuciones)
'xyzxyz': tiempo medio = 76.6460 ms (30 ejecuciones)


**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [8]:
from collections import defaultdict

def construir_indice_invertido(documentos_tokenizados):
    """Vocabulario -> conjunto de ids de documentos donde aparece cada término."""
    indice = defaultdict(set)
    for doc_id, tokens in documentos_tokenizados.items():
        for token in tokens:
            indice[token].add(doc_id)
    return indice

indice_invertido = construir_indice_invertido(documentos_tokenizados)
print(f"Tamaño del vocabulario: {len(indice_invertido)}")

# Inspección de postings y frecuencia documental (DF)
for t in terminos_prueba:
    postings = indice_invertido.get(t, set())
    print(f"'{t}': DF = {len(postings)} -> postings: {sorted(postings)}")

# --- Consulta de un término ---
def consulta_un_termino(termino, indice):
    return indice.get(termino.lower(), set())

# --- Consultas booleanas ---
def consulta_and(t1, t2, indice):
    return consulta_un_termino(t1, indice) & consulta_un_termino(t2, indice)

def consulta_or(t1, t2, indice):
    return consulta_un_termino(t1, indice) | consulta_un_termino(t2, indice)

def consulta_not(t1, indice, todos_los_docs):
    return set(todos_los_docs) - consulta_un_termino(t1, indice)

todos_los_docs = set(documentos_tokenizados.keys())

print("\nAND('the', 'whale'):", consulta_and("the", "whale", indice_invertido))
print("OR('whale', 'xyzxyz'):", consulta_or("whale", "xyzxyz", indice_invertido))
print("NOT('whale'):", len(consulta_not("whale", indice_invertido, todos_los_docs)), "documentos")

Tamaño del vocabulario: 41509
'the': DF = 18 -> postings: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
'whale': DF = 6 -> postings: ['bible-kjv.txt', 'bryant-stories.txt', 'chesterton-ball.txt', 'melville-moby_dick.txt', 'shakespeare-hamlet.txt', 'whitman-leaves.txt']
'xyzxyz': DF = 0 -> postings: []

AND('the', 'whale'): {'bryant-stories.txt', 'melville-moby_dick.txt', 'chesterton-ball.txt', 'bible-kjv.txt', 'whitman-leaves.txt', 'shakespeare-hamlet.txt'}
OR('whale', 'xyzxyz'): {'bryant-stories.txt', 'melville-moby_dick.txt', 'bible-kjv.txt', 'whitman-leaves.txt', 'shakespeare-hamlet.txt', 'chesterton-ball.txt'}
NOT(

**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [9]:
def medir_tiempo_indice(termino, indice, n_ejecuciones=30):
    """Promedia el tiempo (en ms) de 30 ejecuciones de consulta_un_termino."""
    tiempos_ns = []
    for _ in range(n_ejecuciones):
        inicio = time.perf_counter_ns()
        consulta_un_termino(termino, indice)
        fin = time.perf_counter_ns()
        tiempos_ns.append(fin - inicio)
    media_ns = sum(tiempos_ns) / len(tiempos_ns)
    return media_ns / 1e6

print(f"{'Término':10s} | {'Secuencial (ms)':16s} | {'Índice (ms)':12s} | ¿Coinciden?")
for t in terminos_prueba:
    r_sec = set(busqueda_secuencial(t, documentos_tokenizados))
    r_idx = consulta_un_termino(t, indice_invertido)
    t_sec = medir_tiempo_secuencial(t, documentos_tokenizados)
    t_idx = medir_tiempo_indice(t, indice_invertido)
    print(f"{t:10s} | {t_sec:16.4f} | {t_idx:12.4f} | {r_sec == r_idx}")

Término    | Secuencial (ms)  | Índice (ms)  | ¿Coinciden?
the        |           0.0157 |       0.0008 | True
whale      |          55.4848 |       0.0015 | True
xyzxyz     |         104.1115 |       0.0014 | True


### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

In [11]:
import os
import re
import tarfile

# Ruta real del archivo descargado
RUTA_TAR = r"C:\Users\Jhonn22\Downloads\txt-files.tar"

# Carpeta donde se extraerán los archivos
DIR_DESTINO = r"C:\Users\Jhonn22\Downloads\gutenberg_extraido"

print("Archivo encontrado:", os.path.exists(RUTA_TAR))

os.makedirs(DIR_DESTINO, exist_ok=True)

# Extraer solamente si no se hizo anteriormente
bandera = os.path.join(DIR_DESTINO, "_extraido.flag")

if not os.path.exists(bandera):
    print("Extrayendo archivo TAR... Esto puede tardar varios minutos.")

    with tarfile.open(RUTA_TAR, "r") as archivo_tar:
        archivo_tar.extractall(DIR_DESTINO)

    with open(bandera, "w", encoding="utf-8") as archivo:
        archivo.write("ok")

    print("Extracción finalizada.")
else:
    print("Los documentos ya fueron extraídos.")

# Localizar todos los archivos TXT
archivos_txt = []

for raiz, carpetas, archivos in os.walk(DIR_DESTINO):
    for nombre in archivos:
        if nombre.lower().endswith(".txt"):
            archivos_txt.append(os.path.join(raiz, nombre))

print(f"Archivos .txt encontrados: {len(archivos_txt)}")


def id_gutenberg(ruta_archivo):
    base = os.path.basename(ruta_archivo)
    coincidencia = re.match(r"(\d+)", base)
    return coincidencia.group(1) if coincidencia else base


# Conserva solamente una versión por identificador
libros_por_id = {}

for ruta in archivos_txt:
    gutenberg_id = id_gutenberg(ruta)

    if gutenberg_id not in libros_por_id:
        libros_por_id[gutenberg_id] = ruta

print(f"Documentos válidos sin duplicados: {len(libros_por_id)}")

Archivo encontrado: False
Extrayendo archivo TAR... Esto puede tardar varios minutos.


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Jhonn22\\Downloads\\txt-files.tar'